In [30]:
import matplotlib.pyplot as plt
from mysql.connector import Error

In [7]:
import mysql.connector

CONFIG = {
    "host": "127.0.0.1",  
    "port": 3306,          
    "user": "TU_USUARIO",,        
    "password": "TU_PASSWORD",        
    "database": "world",   
}



In [2]:
#  Ejercicio 1: Escribe una consulta para mostrar el nombre y la población de todos los países del continente europeo.
QUERY_PAISES_EUROPA = """
SELECT name, population 
FROM country 
WHERE continent = 'Europe'
ORDER BY population DESC;
"""
# Ejercicio 2: Escribe una consulta para mostrar los nombres y las áreas de superficie de los cinco países más grandes del mundo (en términos de área de superficie).
QUERY_PAISES_MAS_GRANDES = """
SELECT name, surfacearea
FROM country
ORDER BY surfacearea DESC
LIMIT 5;
"""
# Ejercicio 3: Escribe una consulta para calcular la población total de todos los países de cada continente y mostrar el resultado junto con el nombre del continente.
QUERY_POBLACION_POR_CONTINENTE = """
SELECT continent, SUM(population) as poblacion_total
FROM country
GROUP BY continent
ORDER BY poblacion_total DESC;
"""

# Ejercicio 4: Escribe una consulta para mostrar el nombre de las ciudades y la población de todos los países de Europa, ordenados por población de la ciudad de manera descendente.
QUERY_CIUDADES_EUROPA = """
SELECT c.name, c.population, co.name as pais
FROM city c
INNER JOIN country co ON c.countrycode = co.code
WHERE co.continent = 'Europe'
ORDER BY c.population DESC;
"""

# Ejercicio 5: Actualiza la población de China (código de país 'CHN') a 1500000000 (1.5 mil millones).
QUERY_ACTUALIZAR_CHINA = """
UPDATE country
SET population = 1500000000
WHERE code = 'CHN';
"""

QUERY_VERIFICAR_CHINA = """
SELECT code, name, population
FROM country
WHERE code = 'CHN';
"""

# Ejercicio 6: Consulta los idiomas oficiales en Sudamérica y gráfica cuántos países comparten cada idioma oficial.
QUERY_IDIOMAS_SUDAMERICA = """
SELECT cl.language, COUNT(DISTINCT cl.countrycode) as cantidad_paises
FROM countrylanguage cl
INNER JOIN country c ON cl.countrycode = c.code
WHERE c.continent = 'South America' AND cl.isofficial = 'T'
GROUP BY cl.language
ORDER BY cantidad_paises DESC;
"""

# Ejercicio 7 Obtén todos los países con esperanza de vida > 75 años y crea un histograma de su distribución.
QUERY_ESPERANZA_VIDA = """ 
SELECT name, LifeExpectancy
FROM country
WHERE LifeExpectancy > 75 
"""

# Ejercicio 8: Calcula la densidad poblacional de todos los países y muestra un gráfico de dispersión entre superficie y población con el color como función del continente.
QUERY_DENSIDAD_POBLACIONAL = """
SELECT 
    name, 
    continent, 
    population, 
    surfacearea, 
    (population / surfacearea) as densidad
FROM country
WHERE population > 0 AND surfacearea > 0;"""

# Ejercicio 9: Visualiza las ciudades con más de 5 millones de habitantes en un gráfico horizontal de barras.
QUERY_CIUDADES_5MILLONES_HABITANTES = """
SELECT name,population
FROM city
WHERE population > 5000000
ORDER BY population ASC;
"""

# Ejercicio 10: Gráfica cuántos idiomas se hablan por continente usando un gráfico de pastel.
QUERY_IDIOMAS_POR_CONTINENTE = """
SELECT c.Continent, COUNT(DISTINCT cl.Language) as cantidad_idiomas
FROM country c
JOIN countrylanguage cl ON c.Code = cl.CountryCode
GROUP BY c.Continent
ORDER BY cantidad_idiomas DESC;
"""


# Esperanza de vida inferior al promedio
QUERY_ESPERANZA_VIDA_BAJO_PROMEDIO = """
SELECT 
    continent,
    COUNT(*) AS paises_afectados,
    ROUND(AVG(lifeexpectancy), 2) AS esperanza_media
FROM country
WHERE lifeexpectancy < (
    SELECT AVG(lifeexpectancy)
    FROM country
    WHERE lifeexpectancy IS NOT NULL
)
AND lifeexpectancy IS NOT NULL
GROUP BY continent
ORDER BY paises_afectados DESC;
"""

# Ejercicio 12, porcentaje de la poblacion total en un pais que viva en la ciudad mas grande
QUERY_PORCENTAJE_CIUDAD_MAS_GRANDE = """
SELECT 
    co.name AS pais,
    c.name AS ciudad_principal,
    ROUND((c.population / co.population) * 100, 2) AS porcentaje
FROM city c
JOIN country co ON c.countrycode = co.code
WHERE c.population = (
    SELECT MAX(population)
    FROM city
    WHERE countrycode = co.code
)
ORDER BY porcentaje DESC
LIMIT 5;
"""

# Ejercicio 13A
# Objetivo:
# Identificar países con alta población donde la diversidad lingüística
# puede suponer una barrera para la educación, la inclusión social
# y la implementación de políticas públicas.
QUERY_PAISES_SIN_IDIOMA_OFICIAL = """
SELECT 
    c.name AS pais,
    c.population,
    COUNT(DISTINCT cl.language) AS total_idiomas
FROM country c
LEFT JOIN countrylanguage cl ON c.code = cl.countrycode
WHERE c.population > 10000000
GROUP BY c.code
HAVING COUNT(CASE WHEN cl.isofficial = 'T' THEN 1 END) = 0
ORDER BY c.population DESC
LIMIT 5;
"""

#Ejercicio 13B
QUERY_PAISES_FRAGMENTACION_LINGUISTICA = """
SELECT 
    c.code,
    c.name,
    c.population,
    c.continent,
    COUNT(DISTINCT cl.language) as total_idiomas,
    COUNT(DISTINCT CASE WHEN cl.isofficial = 'T' THEN cl.language END) as idiomas_oficiales,
    GROUP_CONCAT(DISTINCT CASE WHEN cl.isofficial = 'T' THEN cl.language END SEPARATOR ', ') as idiomas_oficiales_lista
FROM country c
LEFT JOIN countrylanguage cl ON c.code = cl.countrycode
WHERE c.population > 10000000
GROUP BY c.code, c.name, c.population, c.continent
HAVING COUNT(DISTINCT cl.language) >= 5
ORDER BY total_idiomas DESC;
"""

# EJERCICIO 13C – Resumen Ejecutivo
# Consolida los países con mayor riesgo lingüístico en una sola tabla,
# clasificándolos según el tipo de problema:
# - Sin idioma oficial
# - Alta fragmentación lingüística
# Pensado como salida directa para toma de decisiones en ONG o instituciones.
QUERY_PAISES_PROBLEMAS_LINGUISTICOS = """
SELECT 
    c.code,
    c.name,
    c.population,
    c.continent,
    COUNT(DISTINCT cl.language) as total_idiomas,
    COUNT(DISTINCT CASE WHEN cl.isofficial = 'T' THEN cl.language END) as idiomas_oficiales,
    GROUP_CONCAT(DISTINCT CASE WHEN cl.isofficial = 'T' THEN cl.language END SEPARATOR ', ') as idiomas_oficiales_lista,
    CASE 
        WHEN COUNT(DISTINCT CASE WHEN cl.isofficial = 'T' THEN cl.language END) = 0 THEN 'Sin idioma oficial'
        WHEN COUNT(DISTINCT cl.language) >= 5 THEN 'Alta fragmentación'
        ELSE 'Mixto'
    END as problema
FROM country c
LEFT JOIN countrylanguage cl ON c.code = cl.countrycode
WHERE c.population > 10000000
GROUP BY c.code, c.name, c.population, c.continent
HAVING COUNT(DISTINCT CASE WHEN cl.isofficial = 'T' THEN cl.language END) = 0
   OR COUNT(DISTINCT cl.language) >= 5
ORDER BY total_idiomas DESC;
"""


def ejecutar_query(conn, titulo, query):
    """Función auxiliar para ejecutar y mostrar resultados"""
    cursor = None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        resultados = cursor.fetchall()

        print("\n" + "="*80)
        print(f"  {titulo}")
        print("="*80)

        if not resultados:
            print("No se encontraron resultados.")
            return

        # Mostrar resultados
        for fila in resultados:
            print(fila)

    except Error as e:
        print(f"ERROR en {titulo}: {e}")
    finally:
        if cursor:
            cursor.close()


def ejecutar_update(conn, titulo, query):
    """Función para ejecutar UPDATE/INSERT/DELETE"""
    cursor = None
    try:
        cursor = conn.cursor()
        cursor.execute(query)
        conn.commit()  # IMPORTANTE: confirmar los cambios

        print("\n" + "="*80)
        print(f"  {titulo}")
        print("="*80)
        print(f"✓ Registros afectados: {cursor.rowcount}")
        print("="*80 + "\n")

    except Error as e:
        print(f"ERROR en {titulo}: {e}")
        conn.rollback()  # Deshacer si hay error
    finally:
        if cursor:
            cursor.close()


def ejecutar_query_con_grafica(conn, titulo, query):
    """Función para ejecutar SELECT, mostrar resultados y hacer gráfica"""
    cursor = None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        resultados = cursor.fetchall()

        print("\n" + "="*80)
        print(f"  {titulo}")
        print("="*80)

        if not resultados:
            print("No se encontraron resultados.")
            return

        # Mostrar resultados en consola
        for fila in resultados:
            print(fila)

        # Preparar datos para la gráfica
        idiomas = [fila['language'] for fila in resultados]
        cantidades = [fila['cantidad_paises'] for fila in resultados]

        # Crear la gráfica
        plt.figure(figsize=(10, 6))
        plt.bar(idiomas, cantidades, color='steelblue', edgecolor='black')
        plt.xlabel('Idioma', fontsize=12, fontweight='bold')
        plt.ylabel('Cantidad de Países', fontsize=12, fontweight='bold')
        plt.title(f'{titulo}', fontsize=14, fontweight='bold')
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    except Error as e:
        print(f"ERROR en {titulo}: {e}")
    finally:
        if cursor:
            cursor.close()


def ejecutar_query_histograma(conn, titulo, query, columna_valor, bins=10):
    """Ejecuta SELECT, imprime tabla resumen y crea histograma profesional"""
    cursor = None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        resultados = cursor.fetchall()

        print("\n" + "="*80)
        print(f"  {titulo}")
        print("="*80)

        if not resultados:
            print("No se encontraron resultados.")
            return

        # 1. TABLA RESUMEN PARA LA CONSOLA (Sin diccionarios {} )
        resultados_top = sorted(resultados, key=lambda x: x[columna_valor], reverse=True)
        print(f"Total de países encontrados: {len(resultados)}")
        print(f"{'País':<30} | {'Esperanza de Vida':<15}")
        print("-" * 50)
        for fila in resultados_top[:10]:
            print(f"{fila['name']:<30} | {fila[columna_valor]:>10} años")
        print("...")

        # 2. PREPARACIÓN DEL HISTOGRAMA
        valores = [fila[columna_valor] for fila in resultados if fila[columna_valor] is not None]

        # Configuración de estilo para que combine con tu diapositiva azul
        plt.figure(figsize=(10, 6))
        # Usamos un color celeste que resalte y bordes limpios
        plt.hist(valores, bins=bins, color='#5dade2', edgecolor='white', linewidth=1)
        
        plt.title(f'{titulo}', fontsize=14, fontweight='bold')
        plt.xlabel('Años de Esperanza de Vida', fontsize=12)
        plt.ylabel("Cantidad de Países", fontsize=12)
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        
        # 3. GUARDAR LA IMAGEN (Se guardará en la misma carpeta que tu Jupyter)
        # 'transparent=True' hará que en tu diapositiva azul no se vea el recuadro blanco
        plt.savefig('grafico_ejercicio_7.png', dpi=300, bbox_inches='tight', transparent=True)
        
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"ERROR en {titulo}: {e}")
    finally:
        if cursor:
            cursor.close()


def ejecutar_query_scatter(conn, titulo, query):
    """Ejecuta SELECT y crea un gráfico de dispersión"""
    cursor = None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        resultados = cursor.fetchall()

        if not resultados:
            print("No hay datos para la gráfica.")
            return

        # Extraer datos
        poblacion = [f['population'] for f in resultados]
        area = [f['surfacearea'] for f in resultados]
        continentes = [f['continent'] for f in resultados]
        nombres = [f['name'] for f in resultados]

        # Mapeo de colores por continente
        colores_map = {
            'Asia': 'red', 'Europe': 'blue', 'Africa': 'green',
            'North America': 'orange', 'South America': 'purple',
            'Oceania': 'cyan', 'Antarctica': 'grey'
        }
        colores = [colores_map.get(c, 'black') for c in continentes]

        plt.figure(figsize=(12, 8))

        # Crear scatter plot
        scatter = plt.scatter(area, poblacion, c=colores,
                              alpha=0.6, edgecolors='w', s=100)

        # Configurar escalas logarítmicas (opcional, pero recomendado porque hay países muy pequeños y otros enormes)
        plt.xscale('log')
        plt.yscale('log')

        plt.title(titulo, fontsize=14, fontweight='bold')
        plt.xlabel('Superficie (km²)', fontsize=12)
        plt.ylabel('Población', fontsize=12)

        # Crear una leyenda manual
        import matplotlib.patches as mpatches
        legend_labels = [mpatches.Patch(color=color, label=cont)
                         for cont, color in colores_map.items()]
        plt.legend(handles=legend_labels, title="Continentes",
                   bbox_to_anchor=(1.05, 1), loc='upper left')

        plt.grid(True, which="both", ls="-", alpha=0.2)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"ERROR en Scatter Plot: {e}")
    finally:
        if cursor:
            cursor.close()


def ejecutar_query_ciudades_grandes(conn, titulo, query):
    cursor = None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        resultados = cursor.fetchall()
        if not resultados:
            print("No se encontraron ciudades con más de 5M.")
            return

        nombres = [f['name'] for f in resultados]
        pob = [f['population'] for f in resultados]

        plt.figure(figsize=(12, 8))
        plt.barh(nombres, pob, color='salmon', edgecolor='black')
        plt.title(titulo, fontweight='bold')
        plt.xlabel('Población')
        # Evita notación científica
        plt.ticklabel_format(style='plain', axis='x')
        plt.grid(axis='x', linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.show()  # <--- Aquí enseña el gráfico
    except Error as e:
        print(f"Error en Ejercicio 9: {e}")
    finally:
        if cursor:
            cursor.close()

# NUEVA FUNCIÓN EJERCICIO 10 (PASTEL)


def ejecutar_query_pastel(conn, titulo, query):
    cursor = None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        resultados = cursor.fetchall()

        if not resultados:
            return

        etiquetas = [f['Continent'] for f in resultados]
        valores = [f['cantidad_idiomas'] for f in resultados]

        total = sum(valores)

        def formato(pct):
            cantidad = int(round(pct * total / 100))
            return f"{cantidad}\n({pct:.1f}%)"

        plt.figure(figsize=(8,8))

        wedges, texts, autotexts = plt.pie(
            valores,
            labels=etiquetas,
            autopct=formato,
            startangle=90,
            colors=plt.cm.Set3.colors,
            wedgeprops={'edgecolor': 'white'},
            textprops={'fontsize': 11}
        )

        plt.title(titulo, fontsize=14, fontweight='bold')
        plt.axis('equal')

        plt.tight_layout()
        plt.show()

    except Error as e:
        print(f"Error en Ej 10: {e}")

    finally:
        if cursor:
            cursor.close()

def ejecutar_query_megaciudades(conn, titulo, query):
    """Ejecuta SELECT para ODS 11, imprime tabla y crea gráfico de barras horizontales"""
    cursor = None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        resultados = cursor.fetchall()

        print("\n" + "="*80)
        print(f"  {titulo}")
        print("="*80)

        if not resultados:
            print("No se encontraron resultados.")
            return

        # 1. TABLA RESUMEN PARA CONSOLA
        print(f"{'País':<30} | {'Ciudad Principal':<25} | {'% Población':<12}")
        print("-" * 75)
        for f in resultados:
            print(f"{f['pais']:<30} | {f['ciudad_principal']:<25} | {f['porcentaje']:>10}%")

        # 2. PREPARACIÓN DE DATOS
        paises = [f['pais'] for f in resultados]
        porcentajes = [f['porcentaje'] for f in resultados]

        plt.figure(figsize=(10, 6))
        
        # Color azul profesional para que combine con tu presentación
        plt.barh(paises, porcentajes, color='#1f77b4', edgecolor='black')
        
        # --- AJUSTES PARA ELIMINAR EL EXCESO DEL 100% ---
        # Forzamos el límite a 105 solo para que quepa el texto del % sin que se pase el eje visual
        plt.xlim(0, 105) 
        
        # Quitamos los márgenes internos automáticos de Matplotlib
        plt.gca().margins(y=0.1) 
        
        # Ponemos los ticks (marquitas) solo hasta 100
        plt.xticks([0, 20, 40, 60, 80, 100])

        # Añadimos etiquetas de valor
        for i, v in enumerate(porcentajes):
            plt.text(v + 1, i, f'{v}%', va='center', fontweight='bold', color='black')

        plt.xlabel('% de población en la ciudad principal', fontsize=12)
        plt.title(titulo, fontsize=14, fontweight='bold', pad=20)
        plt.gca().invert_yaxis()
        plt.grid(axis='x', linestyle='--', alpha=0.4)
        
        # 3. GUARDAR IMAGEN
        plt.savefig('grafico_megaciudades_final.png', dpi=300, bbox_inches='tight', transparent=True)
        
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error en {titulo}: {e}")
    finally:
        if cursor:
            cursor.close()

def ejecutar_query_fragmentacion_linguistica_mejorado(conn, titulo, query, top_n=10):
    cursor = None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        resultados = cursor.fetchall()
        if not resultados:
            print("No hay datos para mostrar.")
            return

        resultados = resultados[:top_n]

        paises = [f['name'] for f in resultados]  # <--- CORRECCIÓN: f['name'] en vez de f['pais']
        total_idiomas = [f['total_idiomas'] for f in resultados]
        idiomas_oficiales = [f['idiomas_oficiales'] for f in resultados]

        plt.figure(figsize=(12,7))
        bar_width = 0.4
        y = range(len(paises))
        plt.barh(y, total_idiomas, height=bar_width, color='skyblue', label='Total Idiomas', edgecolor='black')
        plt.barh([i + bar_width for i in y], idiomas_oficiales, height=bar_width, color='orange', label='Idiomas Oficiales', edgecolor='black')
        plt.yticks([i + bar_width/2 for i in y], paises)
        plt.xlabel('Cantidad de Idiomas')
        plt.title(titulo, fontsize=14, fontweight='bold')
        plt.legend()
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()

    except Exception as e:  # <--- Agregado para capturar cualquier error y evitar SyntaxError
        print(f"ERROR en {titulo}: {e}")
    finally:
        if cursor:
            cursor.close()

def ejecutar_query_problemas_linguisticos_mejorado(conn, titulo, query, top_n=10):
    cursor = None
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query)
        resultados = cursor.fetchall()
        if not resultados:
            print("No hay datos para mostrar.")
            return

        resultados = resultados[:top_n]

        paises = [f['name'] for f in resultados]  # <--- CORRECCIÓN: f['name'] en vez de f['pais']
        problemas = [f['problema'] for f in resultados]
        colores = ['red' if p=='Sin idioma oficial' else 'orange' for p in problemas]

        plt.figure(figsize=(12,7))
        plt.barh(paises, [1]*len(paises), color=colores, edgecolor='black')
        for i, p in enumerate(problemas):
            plt.text(0.5, i, p, ha='center', va='center', color='white', fontweight='bold')
        plt.yticks(paises)
        plt.xticks([])
        plt.title(titulo, fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

    except Exception as e:  # <--- Agregado para capturar cualquier error y evitar SyntaxError
        print(f"ERROR en {titulo}: {e}")
    finally:
        if cursor:
            cursor.close()

def main():
    conn = None
    try:
        conn = mysql.connector.connect(**CONFIG)

        # Ejercicio 1: Países de Europa
        ejecutar_query(conn, "EJERCICIO 1: Países de Europa",
                       QUERY_PAISES_EUROPA)

        # Ejercicio 2: 5 países más grandes
        ejecutar_query(
            conn, "EJERCICIO 2: Top 5 Países más Grandes (por área)", QUERY_PAISES_MAS_GRANDES)

        # Ejercicio 3: Población por continente
        ejecutar_query(conn, "EJERCICIO 3: Población Total por Continente",
                       QUERY_POBLACION_POR_CONTINENTE)

        # Ejercicio 4: Ciudades de Europa
        ejecutar_query(conn, "EJERCICIO 4: Ciudades de Europa",
                       QUERY_CIUDADES_EUROPA)

        # Ejercicio 5: Actualizar población de China
        ejecutar_update(
            conn, "EJERCICIO 5: Actualizar Población de China", QUERY_ACTUALIZAR_CHINA)

        # Verificar que se actualizó correctamente
        ejecutar_query(
            conn, "Verificación: Población de China Actualizada", QUERY_VERIFICAR_CHINA)

        # Ejercicio 6: Idiomas oficiales en Sudamérica
        ejecutar_query_con_grafica(
            conn, "EJERCICIO 6: Idiomas Oficiales en Sudamérica", QUERY_IDIOMAS_SUDAMERICA)

        # Ejercicio 7: Esperanza de vida > 75 años
        ejecutar_query_histograma(
            conn,
            "EJERCICIO 7: Esperanza de Vida > 75 años",
            QUERY_ESPERANZA_VIDA,
            columna_valor='LifeExpectancy',
            bins=10
        )

        # Ejercicio 8: Densidad poblacional
        ejecutar_query_scatter(
            conn, "EJERCICIO 8: Dispersión Población vs Superficie", QUERY_DENSIDAD_POBLACIONAL)

        # Ejercico 9: Ciudades con mas de 5 millones de habitantes
        ejecutar_query_ciudades_grandes(
            conn, "EJERCICIO 9: Ciudades con + 5 Millones de Habitantes", QUERY_CIUDADES_5MILLONES_HABITANTES)

        # EJERCICIO 10 idiomas por continente
        ejecutar_query_pastel(
            conn, "EJERCICIO 10: Idiomas por Continente", QUERY_IDIOMAS_POR_CONTINENTE)
        # --------------------------

       # EJERCICIO 11 – Salud
        ejecutar_query(
            conn,"EJERCICIO 11: Regiones con Esperanza de Vida Bajo Promedio (ODS 3)",QUERY_ESPERANZA_VIDA_BAJO_PROMEDIO)

       # EJERCICIO 12 – Megaciudades
        ejecutar_query_megaciudades(
            conn,"EJERCICIO 12: Países con Mayor Riesgo por Megaciudades (ODS 11)",QUERY_PORCENTAJE_CIUDAD_MAS_GRANDE)

        # EJERCICIO 13A – Idioma oficial
        ejecutar_query(
            conn,"EJERCICIO 13A: Países sin Idioma Oficial (Alta Población)",QUERY_PAISES_SIN_IDIOMA_OFICIAL)

       # EJERCICIO 13B – Fragmentación lingüística
        ejecutar_query_fragmentacion_linguistica_mejorado(conn, "EJERCICIO 13B: Top 10 Países con Fragmentación Lingüística", QUERY_PAISES_FRAGMENTACION_LINGUISTICA)

        # Ejercicio 13c: Combinado
        ejecutar_query_problemas_linguisticos_mejorado(conn, "EJERCICIO 13C: Países con Riesgo Lingüístico (Top 10)", QUERY_PAISES_PROBLEMAS_LINGUISTICOS)

        print("\n" + "="*80)
        print("  ✓ Todas las consultas ejecutadas correctamente")
        print("="*80 + "\n")

    except Error as e:
        print("ERROR de conexión:", e)
    finally:
        if conn and conn.is_connected():
            conn.close()


if __name__ == "__main__":
    main()



NameError: name 'Error' is not defined